<a href="https://colab.research.google.com/github/tarunjagrit/HTML-CSS-portfolio/blob/main/PancRisk_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hello")

hello


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix


In [ ]:
df = pd.read_excel("journal.pmed.1003489.s009.xlsx", sheet_name="SAMPLES")

df.head(10)


,Sample ID,Patient's Cohort,Sample Origin,Age,Sex,"Diagnosis (1=Control, 2=Benign, 3=PDAC)",Stage,Benign Samples Diagnosis,Plasma CA19-9 U/ml,Creatinine mg/ml,LYVE1 ng/ml,REG1B ng/ml,TFF1 ng/ml,REG1A ng/ml
0,S1,Cohort1,BPTB,33.0,F,1.0,NaN,NaN,11.7,1.83222,0.893219,52.94884,654.282174,1262.000
1,S10,Cohort1,BPTB,81.0,F,1.0,NaN,NaN,NaN,0.97266,2.037585,94.46703,209.488250,228.407
2,S100,Cohort2,BPTB,51.0,M,1.0,NaN,NaN,7.0,0.78039,0.145589,102.36600,461.141000,NaN
3,S101,Cohort2,BPTB,61.0,M,1.0,NaN,NaN,8.0,0.70122,0.002805,60.57900,142.950000,NaN
4,S102,Cohort2,BPTB,62.0,M,1.0,NaN,NaN,9.0,0.21489,0.000860,65.54000,41.088000,NaN
5,S103,Cohort2,BPTB,53.0,M,1.0,NaN,NaN,NaN,0.84825,0.003393,62.12600,59.793000,NaN
6,S104,Cohort2,BPTB,70.0,M,1.0,NaN,NaN,NaN,0.62205,0.174381,152.27700,117.516000,NaN
7,S105,Cohort2,BPTB,58.0,F,1.0,NaN,NaN,11.0,0.89349,0.003574,3.73000,40.294000,NaN
8,S106,Cohort2,BPTB,59.0,F,1.0,NaN,NaN,NaN,0.48633,0.001945,7.02100,26.782000,NaN
9,S107,Cohort2,BPTB,56.0,F,1.0,NaN,NaN,24.0,0.61074,0.278778,83.92800,19.185000,NaN


In [ ]:
df.columns


Index(['Sample ID', 'Patient's Cohort', 'Sample Origin', 'Age', 'Sex',
       'Diagnosis (1=Control, 2=Benign, 3=PDAC)', 'Stage',
       'Benign Samples Diagnosis', 'Plasma CA19-9 U/ml', 'Creatinine mg/ml',
       'LYVE1 ng/ml', 'REG1B ng/ml', 'TFF1 ng/ml', 'REG1A ng/ml'],
      dtype='object')

In [ ]:
df.isnull().sum()


,0
Sample ID,1
Patient's Cohort,6
Sample Origin,6
Age,6
Sex,6
"Diagnosis (1=Control, 2=Benign, 3=PDAC)",6
Stage,397
Benign Samples Diagnosis,388
Plasma CA19-9 U/ml,246
Creatinine mg/ml,6


In [ ]:
df["Diagnosis (1=Control, 2=Benign, 3=PDAC)"].value_counts()


,count
"Diagnosis (1=Control, 2=Benign, 3=PDAC)",
2.0,208
3.0,199
1.0,183


In [ ]:
df["is_PDAC"] = (df["Diagnosis (1=Control, 2=Benign, 3=PDAC)"] == 3).astype(int)

In [ ]:
df["is_PDAC"].value_counts()

,count
is_PDAC,
0,397
1,199


**FEATURE EXTRACTION**

In [ ]:
features=["Age","LYVE1 ng/ml","REG1B ng/ml","TFF1 ng/ml","Creatinine mg/ml"]
X=df[features]
y=df["is_PDAC"]

In [ ]:
df[features].head()

,Age,LYVE1 ng/ml,REG1B ng/ml,TFF1 ng/ml,Creatinine mg/ml
0,33.0,0.893219,52.94884,654.282174,1.83222
1,81.0,2.037585,94.46703,209.488250,0.97266
2,51.0,0.145589,102.36600,461.141000,0.78039
3,61.0,0.002805,60.57900,142.950000,0.70122
4,62.0,0.000860,65.54000,41.088000,0.21489


**TRUE VALUES**

In [ ]:
df["is_PDAC"].head(10)


,is_PDAC
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0


In [ ]:
df["is_PDAC"].value_counts()


,count
is_PDAC,
0,397
1,199


In [ ]:
X.head()

,Age,LYVE1 ng/ml,REG1B ng/ml,TFF1 ng/ml,Creatinine mg/ml
0,33.0,0.893219,52.94884,654.282174,1.83222
1,81.0,2.037585,94.46703,209.488250,0.97266
2,51.0,0.145589,102.36600,461.141000,0.78039
3,61.0,0.002805,60.57900,142.950000,0.70122
4,62.0,0.000860,65.54000,41.088000,0.21489


In [ ]:
X.describe()


,Age,LYVE1 ng/ml,REG1B ng/ml,TFF1 ng/ml,Creatinine mg/ml
count,590.000000,590.000000,590.000000,590.000000,590.000000
mean,59.079661,3.063530,111.774090,597.868722,0.855383
std,13.109520,3.438796,196.267110,1010.477245,0.639028
min,26.000000,0.000129,0.001104,0.005293,0.056550
25%,50.000000,0.167179,10.757216,43.961000,0.373230
50%,60.000000,1.649862,34.303353,259.873974,0.723840
75%,69.000000,5.205037,122.741013,742.736000,1.139482
max,89.000000,23.890323,1403.897600,13344.300000,4.116840


**PREPROCESSING**

In [ ]:
X_log = np.log1p(X)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)


**Combine into full model**

In [ ]:
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("prep", preprocess),
    ("clf", clf)
])


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = cross_val_score(
    model, X_log, y,
    cv=cv,
    scoring="roc_auc"
)

print("Fold AUCs:", auc_scores)
print("Mean AUC:", auc_scores.mean())


Fold AUCs: [0.91875    0.89391026 0.87753165 0.83481013 0.88386076]
Mean AUC: 0.8817725576111652


In [ ]:
model.fit(X_log, y)

coef = model.named_steps["clf"].coef_[0]
for f, c in zip(features, coef):
    print(f, round(c, 3))


Age 0.691
LYVE1 ng/ml 1.231
REG1B ng/ml 0.478
TFF1 ng/ml 0.327
Creatinine mg/ml -0.514


**INCORPORATING CA19-9**

In [ ]:
df["Plasma CA19-9 U/ml"].isnull().sum(), df.shape[0]


(np.int64(246), 596)

In [ ]:
df["Plasma CA19-9 U/ml"].describe()


,Plasma CA19-9 U/ml
count,350.000000
mean,654.002944
std,2430.317642
min,0.000000
25%,8.000000
50%,26.500000
75%,294.000000
max,31000.000000


In [ ]:
features_ca = ["LYVE1 ng/ml", "REG1B ng/ml", "TFF1 ng/ml", "Creatinine mg/ml", "Age", "Plasma CA19-9 U/ml"]
X_ca = df[features_ca]
y = df["is_PDAC"]


In [ ]:
X_ca_log = np.log1p(X_ca)


In [ ]:
auc_ca = cross_val_score(model, X_ca_log, y, cv=cv, scoring="roc_auc")

print("PancRISK + CA19-9 AUCs:", auc_ca)
print("Mean AUC:", auc_ca.mean())


PancRISK + CA19-9 AUCs: [0.98125    0.92724359 0.9335443  0.94398734 0.89588608]
Mean AUC: 0.9363822622525154


In [ ]:
df_cb = df[df["Diagnosis (1=Control, 2=Benign, 3=PDAC)"].isin([2,3])].copy()
df_cb["y_cb"] = (df_cb["Diagnosis (1=Control, 2=Benign, 3=PDAC)"] == 3).astype(int)

X_cb_base = np.log1p(df_cb[features])
X_cb_ca   = np.log1p(df_cb[features_ca])
y_cb = df_cb["y_cb"]

auc_cb_base = cross_val_score(model, X_cb_base, y_cb, cv=cv, scoring="roc_auc")
auc_cb_ca   = cross_val_score(model, X_cb_ca,   y_cb, cv=cv, scoring="roc_auc")

print("Baseline (urine only):", auc_cb_base.mean())
print("With CA19-9:", auc_cb_ca.mean())


Baseline (urine only): 0.8503290002680245
With CA19-9: 0.90305056731886


In [ ]:
model.fit(X_ca_log, y)
for f, c in zip(features_ca, model.named_steps["clf"].coef_[0]):
    print(f, round(c,3))


LYVE1 ng/ml 1.139
REG1B ng/ml 0.607
TFF1 ng/ml 0.001
Creatinine mg/ml -0.482
Age 0.564
Plasma CA19-9 U/ml 1.933


**EXPERIMENTATION **

> Combination of LYVE1/REG1B/TFF1/Age





In [ ]:
features_ca1= ["LYVE1 ng/ml", "REG1B ng/ml", "TFF1 ng/ml", "Age"]
X_exp1 = df[features_ca1]
y = df["is_PDAC"]


In [ ]:
X_exp1_log=np.log1p(X_exp1)

In [ ]:
auc_X_exp1 = cross_val_score(model, X_exp1_log, y, cv=cv, scoring="roc_auc")

In [ ]:
print("Combination of LYVE1/REG1B/TFF1/Age", auc_X_exp1)
print("Mean combo1  AUC:", auc_X_exp1.mean())

Combination of LYVE1/REG1B/TFF1/Age [0.920625   0.89070513 0.86297468 0.81962025 0.87626582]
Mean combo1  AUC: 0.8740381775397598


EXPERIMENTATION 2

Combination of LYVE1/REG1B/Age



In [ ]:
features_com2 = ["LYVE1 ng/ml", "REG1B ng/ml", "Age"]
X_exp2 = df[features_com2]
y = df["is_PDAC"]

In [ ]:
X_exp2_log=np.log1p(X_exp2)

In [ ]:
auc_X_exp2 = cross_val_score(model, X_exp2_log, y, cv=cv, scoring="roc_auc")

In [ ]:
print("Combination of LYVE1/REG1B/Age", auc_X_exp2)
print("Mean combo2  AUC:", auc_X_exp2.mean())

Combination of LYVE1/REG1B/Age [0.9196875  0.89294872 0.86265823 0.8193038  0.87943038]
Mean combo2  AUC: 0.8748057246024018


EXPERIMENTATION 3

Combination of LYVE1 and REG1B only


In [ ]:
features_com3 = ["LYVE1 ng/ml", "REG1B ng/ml"]
X_exp3 = df[features_com3]
y = df["is_PDAC"]

In [ ]:
X_exp3_log=np.log1p(X_exp3)

In [ ]:
auc_X_exp3 = cross_val_score(model, X_exp3_log, y, cv=cv, scoring="roc_auc")

In [ ]:
print("Combination of LYVE1 and REG1B", auc_X_exp3)
print("Mean combo3  AUC:", auc_X_exp3.mean())

Combination of LYVE1 and REG1B [0.905625   0.85673077 0.83132911 0.83607595 0.85126582]
Mean combo3  AUC: 0.8562053310613438


EXPERIMENTATION 4

Combination of LYVE1 ONLY

In [ ]:
features_com4 = ["LYVE1 ng/ml"]
X_exp4 = df[features_com4]
y = df["is_PDAC"]

In [ ]:
X_exp4_log=np.log1p(X_exp4)

In [ ]:
auc_X_exp4 = cross_val_score(model, X_exp4_log, y, cv=cv, scoring="roc_auc")

In [ ]:
print("Combination of LYVE1 ", auc_X_exp4)
print("Mean combo4  AUC:", auc_X_exp4.mean())

Combination of LYVE1  [0.886875   0.84839744 0.82025316 0.83006329 0.84841772]
Mean combo4  AUC: 0.846801322622525


EXPERIMENTATION 5

Combination of LYVE1 and CA 19-9

In [ ]:
features_com5 = ["LYVE1 ng/ml","Plasma CA19-9 U/ml"]
X_exp5 = df[features_com5]
y = df["is_PDAC"]

In [ ]:
X_exp5_log=np.log1p(X_exp5)

In [ ]:
auc_X_exp5 = cross_val_score(model, X_exp5_log, y, cv=cv, scoring="roc_auc")

In [ ]:
print("Combination of LYVE1 and CA19-9", auc_X_exp5)
print("Mean combo5  AUC:", auc_X_exp5.mean())

Combination of LYVE1 and CA19-9 [0.9703125  0.90769231 0.91487342 0.94620253 0.86962025]
Mean combo5  AUC: 0.9217402020447907


In [ ]:
from sklearn.model_selection import cross_val_predict

y_prob_exp5 = cross_val_predict(
    model,
    X_exp5,          # LYVE1 + CA19-9 features
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]


In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

fpr, tpr, thresholds = roc_curve(y, y_prob_exp5)
specificity = 1 - fpr

def sens_at_spec(spec_target):
    idx = np.where(specificity >= spec_target)[0][-1]
    return tpr[idx], thresholds[idx]

sens90, thr90 = sens_at_spec(0.90)
sens95, thr95 = sens_at_spec(0.95)


In [ ]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(y, y_prob_exp5)


In [ ]:
print("Combination: LYVE1 + CA19-9")
print("AUCs:", auc_X_exp5)
print("Mean AUC:", round(auc_X_exp5.mean(), 3))

print("Sensitivity @ 90% specificity:", round(sens90, 3))
print("Sensitivity @ 95% specificity:", round(sens95, 3))


print("Brier score:", round(brier, 3))


Combination: LYVE1 + CA19-9
AUCs: [0.9703125  0.90769231 0.91487342 0.94620253 0.86962025]
Mean AUC: 0.922
Sensitivity @ 90% specificity: 0.688
Sensitivity @ 95% specificity: 0.523
Brier score: 0.12
